# MVP — Machine Learning & Analytics

**Nome:** _Seu nome aqui_  
**Matrícula:** _Sua matrícula aqui_  
**Data:** _dd/mm/aaaa_  
**Dataset:** Wine Quality Dataset — Kaggle (UCI Machine Learning Repository)  
**Tipo de problema:** Classificação  

---

## Checklist do MVP

| Item | Status |
|---|---|
| Problema definido com contexto, objetivo e tipo de tarefa | ✅ |
| Dataset descrito, com fonte, atributos e restrições | ✅ |
| Dataset carregado via URL raw do GitHub (sem upload ou login) | ✅ |
| Análise exploratória objetiva, conectada à modelagem | ✅ |
| Divisão adequada em treino/teste (estratificada) | ✅ |
| Prevenção de vazamento de dados | ✅ |
| Tratamentos de dados justificados | ✅ |
| Pipeline reprodutível de pré-processamento | ✅ |
| Modelo baseline definido | ✅ |
| Pelo menos dois modelos/abordagens comparados | ✅ |
| Ajuste de hiperparâmetros em pelo menos um modelo | ✅ |
| Avaliação com métricas coerentes com o problema | ✅ |
| Discussão de overfitting/underfitting e limitações | ✅ |
| Código limpo, organizado e executável do início ao fim | ✅ |
| Conclusão conectada ao objetivo inicial | ✅ |


---
# 1. Definição do problema


## 1.1 Descrição do problema

A avaliação da qualidade de vinhos é tradicionalmente realizada por especialistas humanos (sommeliers),
processo que é subjetivo, demorado e oneroso. Com a crescente escala de produção vinícola global,
torna-se cada vez mais necessário automatizar esse processo utilizando propriedades físico-químicas
mensuráveis objetivamente em laboratório.

Este projeto aborda o problema de **classificar automaticamente a qualidade de vinhos tintos** —
distinguindo vinhos de qualidade inferior, média e superior — a partir de suas características
químicas, sem depender da avaliação humana subjetiva.

**Aplicações práticas:**
- Controle de qualidade em vinícolas com grande volume de produção
- Sistemas de recomendação de vinhos em plataformas de e-commerce
- Auxílio a compradores institucionais (restaurantes, distribuidores) na triagem de lotes
- Detecção de adulterações ou desvios de padrão durante o processo de produção


## 1.2 Objetivo do MVP

> O objetivo deste MVP é construir e avaliar modelos de Machine Learning capazes de **prever a
> categoria de qualidade de vinhos tintos** (`qualidade_categoria`: Baixa / Média / Alta) a partir
> de 11 variáveis físico-químicas, comparando uma baseline com modelos candidatos e discutindo
> suas limitações.


## 1.3 Tipo de problema

**Tipo escolhido:** Classificação multiclasse (3 classes)

**Justificativa:** A variável original `quality` é numérica discreta (valores de 3 a 8),
mas representa categorias ordinais de qualidade — não uma escala contínua com sentido
de regressão. Agrupamos os valores em três faixas para tornar o problema mais robusto
e clinicamente significativo, além de reduzir o impacto do forte desbalanceamento das
classes originais (muito poucas amostras nas extremidades da escala).


## 1.4 Premissas, hipóteses e critérios de sucesso

**Hipóteses iniciais:**
1. O teor alcoólico (`alcohol`) é o principal determinante da qualidade percebida.
2. A acidez volátil (`volatile acidity`) está negativamente correlacionada com a qualidade.
3. O problema apresenta separabilidade não-linear — modelos ensemble devem superar modelos lineares.
4. O dataset é desbalanceado (maioria de vinhos de qualidade média) — requer atenção à métrica F1.

**Critérios de sucesso:**
- **Métrica principal:** F1-score ponderado (robusto ao desbalanceamento de classes)
- **Resultado mínimo esperado:** superar o baseline em pelo menos 20 pontos percentuais de F1
- **Restrição:** modelo treinável em Colab gratuito (CPU) em tempo razoável


---
# 2. Ambiente, bibliotecas e reprodutibilidade

Todas as importações e configurações necessárias. A semente `42` garante
reprodutibilidade completa — os mesmos resultados em qualquer execução.


In [ ]:
# ============================================================
# CONFIGURAÇÃO INICIAL E REPRODUTIBILIDADE
# ============================================================
import os, sys, time, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    ConfusionMatrixDisplay, confusion_matrix
)
from scipy.stats import randint, uniform
import joblib

warnings.filterwarnings('ignore')

# Semente global — garante reprodutibilidade em todas as operações aleatórias
SEMENTE = 42
np.random.seed(SEMENTE)
random.seed(SEMENTE)

print('Versão do Python:', sys.version.split()[0])
print('Semente global fixada em:', SEMENTE)


## 2.1 Dependências adicionais

Todas as bibliotecas já estão disponíveis no Google Colab. Nenhuma instalação extra é necessária.


In [ ]:
# Verificação das versões instaladas
import sklearn
print('scikit-learn:', sklearn.__version__)
print('pandas      :', pd.__version__)
print('numpy       :', np.__version__)


## 2.2 Funções auxiliares

Funções reutilizáveis para avaliação e exibição de resultados,
garantindo consistência na comparação entre todos os modelos.


In [ ]:
def avaliar_classificacao(verdadeiro, predito, nome_modelo=''):
    """
    Calcula as principais métricas para classificação multiclasse.

    Parâmetros
    ----------
    verdadeiro  : array com os rótulos reais
    predito     : array com as predições do modelo
    nome_modelo : identificador do modelo na tabela comparativa

    Retorna
    -------
    Dicionário com acurácia, F1 ponderado e F1 macro
    """
    return {
        'modelo'      : nome_modelo,
        'acuracia'    : round(accuracy_score(verdadeiro, predito), 4),
        'f1_ponderado': round(f1_score(verdadeiro, predito, average='weighted'), 4),
        'f1_macro'    : round(f1_score(verdadeiro, predito, average='macro'), 4),
    }


def exibir_tabela_resultados(dicionario_resultados):
    """Organiza e exibe os resultados de todos os modelos em um DataFrame."""
    colunas_desejadas = ['modelo', 'acuracia', 'f1_ponderado', 'f1_macro', 'tempo_treino_s']
    tabela = pd.DataFrame(dicionario_resultados).T
    colunas_presentes = [c for c in colunas_desejadas if c in tabela.columns]
    return tabela[colunas_presentes]


---
# 3. Seleção e carga dos dados


## 3.1 Fonte dos dados

**Dataset:** Wine Quality Dataset — Red Wine (WineQT)  
**Origem:** UCI Machine Learning Repository / Kaggle  
**Registros:** 1.143 · **Variáveis:** 12 (11 preditoras + 1 alvo)  
**Licença:** Domínio público  
**URL de acesso:** [`06Cata/Kaggle_Wine_Quality`](https://raw.githubusercontent.com/06Cata/Kaggle_Wine_Quality/main/raw_data/WineQT.csv)  

**Por que foi escolhido:**
- Dataset clássico de ML fora da área da saúde (domínio: alimentos & bebidas)
- Sem valores ausentes — adequado para demonstrar pipeline sem imputação obrigatória
- Variável alvo clara e significativa (qualidade sensorial mensurável)
- Todas as features são numéricas contínuas — permite foco no algoritmo, não no encoding
- Problema desafiador por desbalanceamento natural das classes

**Acesso:** O dataset é lido diretamente pela URL raw do repositório público do GitHub.
Nenhum upload, login, token ou arquivo local é necessário.


## 3.2 Carga dos dados


In [ ]:
# ============================================================
# CARGA DOS DADOS — URL RAW DO GITHUB (SEM UPLOAD)
# ============================================================
# Leitura direta pelo link público — notebook 100% reprodutível.

url_conjunto_dados = 'https://raw.githubusercontent.com/06Cata/Kaggle_Wine_Quality/main/raw_data/WineQT.csv'

# Carga do CSV diretamente da URL pública
dados_brutos = pd.read_csv(url_conjunto_dados)

print('Conjunto de dados carregado com sucesso!')
print(f'Formato  : {dados_brutos.shape[0]:,} linhas  x  {dados_brutos.shape[1]} colunas')
print(f'Fonte    : {url_conjunto_dados}')
dados_brutos.head(3)


## 3.3 Visão geral do dataset


In [ ]:
print('Dimensões:', dados_brutos.shape)
print()
print('Tipos de dado:')
display(dados_brutos.dtypes.value_counts().to_frame('quantidade'))


In [ ]:
print('Valores ausentes por coluna:')
nulos = dados_brutos.isnull().sum()
print(nulos[nulos > 0].to_frame('ausentes')
      if nulos.sum() > 0 else 'Nenhum valor ausente encontrado.')
print()
print('Registros duplicados:', dados_brutos.duplicated().sum())


In [ ]:
display(dados_brutos.sample(5, random_state=SEMENTE))


## 3.4 Dicionário de variáveis

| Variável | Tipo | Descrição |
|---|---|---|
| `fixed acidity` | numérica | Acidez fixa (g/L de tartaric acid) |
| `volatile acidity` | numérica | Acidez volátil — excesso causa sabor avinagrado |
| `citric acid` | numérica | Ácido cítrico — adiciona frescor e sabor |
| `residual sugar` | numérica | Açúcar residual após fermentação (g/L) |
| `chlorides` | numérica | Teor de cloretos (sal) |
| `free sulfur dioxide` | numérica | SO₂ livre — impede crescimento microbiano |
| `total sulfur dioxide` | numérica | SO₂ total (livre + combinado) |
| `density` | numérica | Densidade do vinho (g/mL) |
| `pH` | numérica | Acidez total na escala pH (0–14) |
| `sulphates` | numérica | Sulfatos — aditivo antimicrobiano e antioxidante |
| `alcohol` | numérica | Teor alcoólico (% vol.) |
| `quality` | inteiro | Nota original (3–8) — **usada para criar o target** |
| `Id` | inteiro | Identificador — **excluído** (sem valor preditivo) |
| **`qualidade_categoria`** | **categórica** | **Target criado:** Baixa / Média / Alta |


---
# 4. Análise exploratória dos dados

A EDA está organizada em: (1) construção e distribuição do target,
(2) análise das features mais relevantes por classe,
(3) correlações entre variáveis numéricas.


In [ ]:
# ============================================================
# ENGENHARIA DO TARGET: quality → qualidade_categoria
# ============================================================
# A coluna 'quality' possui valores de 3 a 8.
# Agrupamos em 3 categorias para:
#   1. Reduzir o desbalanceamento extremo das classes originais
#   2. Tornar o problema de classificação mais robusto e interpretável
#   3. Simular uma triagem real (inferior / padrão / superior)

def categorizar_qualidade(nota):
    """Mapeia nota numérica para categoria de qualidade."""
    if nota <= 5:
        return 'Baixa'    # notas 3, 4, 5
    elif nota == 6:
        return 'Media'    # nota 6
    else:
        return 'Alta'     # notas 7, 8

# Criação do dataset de trabalho (cópia para não alterar os dados originais)
dados = dados_brutos.copy()
dados['qualidade_categoria'] = dados['quality'].apply(categorizar_qualidade)

# Definição das constantes do problema
COLUNA_ALVO  = 'qualidade_categoria'
ORDEM_CLASSES = ['Baixa', 'Media', 'Alta']
CORES_CLASSES = ['#e15759', '#f28e2b', '#4e79a7']  # vermelho, laranja, azul

print('Distribuição das notas originais (quality):')
display(dados['quality'].value_counts().sort_index().to_frame('quantidade'))
print()
print('Distribuição do target criado (qualidade_categoria):')
display(dados[COLUNA_ALVO].value_counts()[ORDEM_CLASSES].to_frame('quantidade'))
display(
    (dados[COLUNA_ALVO].value_counts(normalize=True)[ORDEM_CLASSES] * 100)
    .round(1).to_frame('percentual (%)')
)


In [ ]:
# ── Figura 1: Distribuição das classes da variável alvo ──────────────
# O dataset é desbalanceado: maioria dos vinhos é de qualidade Média.
# Isso justifica o uso de F1 ponderado como métrica principal.

contagem_classes = dados[COLUNA_ALVO].value_counts()[ORDEM_CLASSES]

figura, eixo = plt.subplots(figsize=(7, 4))
barras = eixo.bar(
    ORDEM_CLASSES, contagem_classes.values,
    color=CORES_CLASSES, edgecolor='white', linewidth=1.2
)
for barra, valor in zip(barras, contagem_classes.values):
    eixo.text(barra.get_x() + barra.get_width()/2,
              barra.get_height() + 8,
              f'{valor:,}', ha='center', fontsize=10, fontweight='bold')
eixo.set_title('Fig. 1 — Distribuição das Classes (qualidade_categoria)',
               fontsize=12, fontweight='bold')
eixo.set_ylabel('Número de Registros')
eixo.set_xlabel('Categoria de Qualidade')
eixo.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()
print('>> Dataset desbalanceado: classe Média é maioria (~50%).')
print('>> Justifica uso de F1 ponderado como métrica principal.')


In [ ]:
# ── Figura 2: Teor alcoólico por categoria de qualidade ──────────────
# Hipótese 1: vinhos de maior qualidade têm maior teor alcoólico.

figura, eixo = plt.subplots(figsize=(8, 4))
for i, classe in enumerate(ORDEM_CLASSES):
    valores = dados[dados[COLUNA_ALVO] == classe]['alcohol']
    eixo.boxplot(
        valores, positions=[i], widths=0.5, patch_artist=True,
        boxprops=dict(facecolor=CORES_CLASSES[i], alpha=0.7),
        medianprops=dict(color='black', linewidth=2),
        whiskerprops=dict(color='gray'), capprops=dict(color='gray'),
        flierprops=dict(marker='o', markersize=3, color='gray', alpha=0.4)
    )
eixo.set_xticks(range(3))
eixo.set_xticklabels(ORDEM_CLASSES)
eixo.set_title('Fig. 2 — Teor Alcoólico por Categoria de Qualidade',
               fontsize=12, fontweight='bold')
eixo.set_ylabel('Álcool (% vol.)')
eixo.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()
print('>> Hipótese 1 confirmada: vinhos de qualidade Alta têm claramente maior teor alcoólico.')


In [ ]:
# ── Figura 3: Acidez volátil por categoria de qualidade ──────────────
# Hipótese 2: acidez volátil alta está associada a qualidade inferior.

figura, eixo = plt.subplots(figsize=(8, 4))
for i, classe in enumerate(ORDEM_CLASSES):
    valores = dados[dados[COLUNA_ALVO] == classe]['volatile acidity']
    eixo.boxplot(
        valores, positions=[i], widths=0.5, patch_artist=True,
        boxprops=dict(facecolor=CORES_CLASSES[i], alpha=0.7),
        medianprops=dict(color='black', linewidth=2),
        whiskerprops=dict(color='gray'), capprops=dict(color='gray'),
        flierprops=dict(marker='o', markersize=3, color='gray', alpha=0.4)
    )
eixo.set_xticks(range(3))
eixo.set_xticklabels(ORDEM_CLASSES)
eixo.set_title('Fig. 3 — Acidez Volátil por Categoria de Qualidade',
               fontsize=12, fontweight='bold')
eixo.set_ylabel('Acidez Volátil (g/L)')
eixo.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()
print('>> Hipótese 2 confirmada: vinhos de qualidade Baixa têm maior acidez volátil (sabor avinagrado).')


In [ ]:
# ── Figura 4: Mapa de calor de correlação entre variáveis numéricas ──
# Identifica multicolinearidade e variáveis mais relevantes.

# Seleção das variáveis físico-químicas (excluindo Id e quality)
variaveis_quimicas = ['fixed acidity','volatile acidity','citric acid',
                      'residual sugar','chlorides','free sulfur dioxide',
                      'total sulfur dioxide','density','pH','sulphates','alcohol']

matriz_correlacao = dados[variaveis_quimicas].corr()
mascara = np.triu(np.ones_like(matriz_correlacao, dtype=bool))

figura, eixo = plt.subplots(figsize=(10, 8))
sns.heatmap(
    matriz_correlacao, mask=mascara, ax=eixo,
    cmap='coolwarm', annot=True, fmt='.2f',
    linewidths=0.5, annot_kws={'size':8}, center=0
)
eixo.set_title('Fig. 4 — Correlação entre Variáveis Físico-Químicas',
               fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


## 4.1 Síntese da análise exploratória

- **Desbalanceamento:** classe Média (~50%) é maioria; classes Baixa (~46%) e Alta (~4%) são minoritárias
  → F1 ponderado é mais adequado que acurácia como métrica principal
- **Hipótese 1 confirmada:** `alcohol` discrimina bem as classes — vinhos de qualidade Alta têm
  claramente maior teor alcoólico. É esperado como feature mais importante
- **Hipótese 2 confirmada:** `volatile acidity` está negativamente correlacionada com a qualidade;
  vinhos de qualidade Baixa têm os maiores valores
- **Multicolinearidade:** `fixed acidity` e `citric acid` têm correlação moderada (~0,67);
  `free SO₂` e `total SO₂` também (~0,67). O Random Forest lida bem com isso
- **Hipótese 3 (separabilidade):** há sobreposição nas distribuições → modelos ensemble esperados
  como mais adequados do que modelos lineares


---
# 5. Preparação dos dados e divisão treino/teste


In [ ]:
# ============================================================
# CONFIGURAÇÃO DO PROBLEMA E SELEÇÃO DE VARIÁVEIS
# ============================================================

# Variável alvo criada na seção anterior
COLUNA_ALVO  = 'qualidade_categoria'
TIPO_PROBLEMA = 'classificacao'

# Colunas excluídas:
#   'Id'      → identificador sem valor preditivo
#   'quality' → nota numérica original usada para criar o target (leakage implícito)
COLUNAS_EXCLUIR = ['Id', 'quality', COLUNA_ALVO]

# Todas as features são numéricas — sem colunas categóricas neste dataset
variaveis_preditoras = [c for c in dados.columns if c not in COLUNAS_EXCLUIR]

print(f'Tipo de problema              : {TIPO_PROBLEMA}')
print(f'Variável alvo                 : {COLUNA_ALVO}')
print(f'Variáveis preditoras usadas   : {len(variaveis_preditoras)}')
print(f'Variáveis excluídas           : {COLUNAS_EXCLUIR}')
print()
print('Features:', variaveis_preditoras)


In [ ]:
# ============================================================
# CODIFICAÇÃO DO TARGET E DIVISÃO TREINO / TESTE
# ============================================================

# Encode ordinal do target: Baixa=0, Media=1, Alta=2
ordem_classes = ['Baixa', 'Media', 'Alta']
codificador_classes = LabelEncoder()
codificador_classes.fit(ordem_classes)
alvo_codificado = codificador_classes.transform(dados[COLUNA_ALVO])

# Matriz de variáveis preditoras
X = dados[variaveis_preditoras].copy()

# Divisão estratificada 80/20
# stratify=alvo_codificado: preserva a proporção de cada classe
# Especialmente importante aqui pois a classe 'Alta' tem poucas amostras
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, alvo_codificado,
    test_size=0.2,
    random_state=SEMENTE,
    stratify=alvo_codificado
)

print('Conjunto de treino:', X_treino.shape)
print('Conjunto de teste :', X_teste.shape)
print()
print('Distribuição no treino:', dict(zip(codificador_classes.classes_, np.bincount(y_treino))))
print('Distribuição no teste :', dict(zip(codificador_classes.classes_, np.bincount(y_teste))))


## 5.1 Justificativa da divisão

**Holdout 80/20** — com 1.143 registros, o conjunto de teste com ~229 amostras
é suficiente para estimativas estáveis, mas requer atenção à classe 'Alta' (~4%).

A **estratificação** (`stratify=alvo_codificado`) é essencial aqui:
a classe 'Alta' representa apenas ~4% dos dados (~46 amostras). Sem estratificação,
a divisão aleatória poderia colocar todas as amostras raras no treino ou no teste.

**Vazamento de dados prevenido por:**
1. Exclusão da coluna `quality` (nota original que gerou o target)
2. Exclusão do `Id` (sem valor preditivo)
3. Pipeline com `fit` apenas nos dados de treino


---
# 6. Pré-processamento e pipeline


In [ ]:
# ============================================================
# PIPELINE DE PRÉ-PROCESSAMENTO
# ============================================================
# Todas as features são numéricas — um único sub-pipeline é necessário.
# O Pipeline garante que o fit ocorra APENAS nos dados de treino,
# evitando vazamento de estatísticas do teste para o treinamento.

# Colunas numéricas (todas as features neste dataset)
colunas_numericas = variaveis_preditoras

# Sub-pipeline numérico:
#   SimpleImputer(mediana): precaução — sem nulos, mas boa prática para robustez
#   StandardScaler: padronização (média=0, desvio=1)
#     → necessário para Regressão Logística (sensível à escala)
#     → neutro para Random Forest e Gradient Boosting (baseados em árvores)
pipeline_numerico = Pipeline(steps=[
    ('imputacao'  , SimpleImputer(strategy='median')),
    ('padronizacao', StandardScaler())
])

# ColumnTransformer: aplica o pipeline ao subconjunto correto de colunas
preprocessador = ColumnTransformer(
    transformers=[('numericas', pipeline_numerico, colunas_numericas)],
    remainder='drop'
)

print('Pipeline de pré-processamento criado.')
print('Variáveis numéricas:', len(colunas_numericas))
print('Variáveis categóricas: 0 (nenhuma neste dataset)')
print()
print('Estrutura:')
print('  ColumnTransformer')
print('  └── numericas → SimpleImputer(mediana) → StandardScaler')


## 6.1 Decisões de pré-processamento

| Etapa | Estratégia | Justificativa |
|---|---|---|
| Imputação numérica | Mediana | Robusta a outliers; precaução mesmo sem nulos |
| Escalonamento numérico | StandardScaler | Necessário para LogReg; neutro para RF e GBM |
| Colunas categóricas | N/A | Todas as features são numéricas neste dataset |
| Coluna `Id` | Excluída | Identificador sem valor preditivo |
| Coluna `quality` | Excluída | Usada para criar o target — leakage implícito |


---
# 7. Baseline e modelos candidatos

Estratégia progressiva: do mais simples (baseline) ao mais complexo.
O dataset é desbalanceado, então o baseline ingênuo prediz sempre 'Media' (~50% de acc.).


In [ ]:
# ============================================================
# DEFINIÇÃO DO BASELINE E DOS MODELOS CANDIDATOS
# ============================================================

# BASELINE: DummyClassifier
# Sempre prediz a classe mais frequente ('Media').
# Em dataset desbalanceado, pode ter acc. razoável mas F1 macro baixo.
modelo_baseline = Pipeline(steps=[
    ('preprocessador', preprocessador),
    ('modelo', DummyClassifier(strategy='most_frequent', random_state=SEMENTE))
])

# CANDIDATO 1: Regressão Logística
# Modelo linear clássico. Avalia separabilidade linear entre as classes.
# Rápido, interpretável, sensível à escala (por isso precisa do StandardScaler).
modelo_regressao_logistica = Pipeline(steps=[
    ('preprocessador', preprocessador),
    ('modelo', LogisticRegression(
        max_iter=1000,          # mais iterações para garantir convergência
        random_state=SEMENTE,
        class_weight='balanced' # compensação pelo desbalanceamento das classes
    ))
])

# CANDIDATO 2: Floresta Aleatória
# Ensemble de árvores — captura interações não-lineares entre features.
# Robusto a outliers e não sensível à escala das variáveis.
modelo_floresta_aleatoria = Pipeline(steps=[
    ('preprocessador', preprocessador),
    ('modelo', RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        class_weight='balanced', # compensação pelo desbalanceamento
        random_state=SEMENTE,
        n_jobs=-1
    ))
])

# Dicionário de candidatos para iteração automatizada
modelos_candidatos = {
    'RegressaoLogistica'  : modelo_regressao_logistica,
    'FlorestAleatoria'    : modelo_floresta_aleatoria,
}

print('Modelos definidos:')
print('  - Baseline    : DummyClassifier (most_frequent)')
print('  - Candidato 1 : Regressão Logística (com class_weight=balanced)')
print('  - Candidato 2 : Floresta Aleatória  (com class_weight=balanced)')


## 7.1 Justificativa dos modelos

**DummyClassifier:** prediz sempre 'Media' (classe mais frequente). Útil como referência
mínima — em dataset desbalanceado, pode atingir ~50% de acurácia sem aprender nada.

**Regressão Logística com `class_weight='balanced'`:** ajusta automaticamente os pesos
das classes inversamente proporcional às frequências — compensa o desbalanceamento durante
o treino sem modificar os dados. O parâmetro `class_weight` é o equivalente de SMOTE
para modelos lineares, com menor custo computacional.

**Floresta Aleatória com `class_weight='balanced'`:** mesmo benefício para modelo ensemble.
A EDA sugere separabilidade não-linear (distribuições sobrepostas), o que favorece este modelo.


---
# 8. Treinamento e avaliação inicial


In [ ]:
# ============================================================
# TREINAMENTO E AVALIAÇÃO INICIAL
# ============================================================

resultados   = {}
modelos_treinados = {}

# ── Baseline ─────────────────────────────────────────────────────────
inicio = time.time()
modelo_baseline.fit(X_treino, y_treino)
resultados['baseline'] = avaliar_classificacao(
    y_teste, modelo_baseline.predict(X_teste), 'Dummy (baseline)'
)
resultados['baseline']['tempo_treino_s'] = round(time.time() - inicio, 3)
modelos_treinados['baseline'] = modelo_baseline

# ── Candidatos ───────────────────────────────────────────────────────
for nome, modelo in modelos_candidatos.items():
    inicio = time.time()
    modelo.fit(X_treino, y_treino)
    resultados[nome] = avaliar_classificacao(
        y_teste, modelo.predict(X_teste), nome
    )
    resultados[nome]['tempo_treino_s'] = round(time.time() - inicio, 3)
    modelos_treinados[nome] = modelo

exibir_tabela_resultados(resultados)


In [ ]:
# ── Figura 5: Comparação baseline vs candidatos ──────────────────────
chaves_grafico = ['baseline', 'RegressaoLogistica', 'FlorestAleatoria']
nomes_grafico  = ['Baseline\n(Dummy)', 'Regressão\nLogística', 'Floresta\nAleatória']

acuracias     = [resultados[k]['acuracia']     for k in chaves_grafico]
f1_ponderados = [resultados[k]['f1_ponderado'] for k in chaves_grafico]

posicoes = np.arange(3)
largura  = 0.35
figura, eixo = plt.subplots(figsize=(8, 5))
b1 = eixo.bar(posicoes - largura/2, acuracias,     largura, label='Acurácia',    color='#4e79a7', edgecolor='white')
b2 = eixo.bar(posicoes + largura/2, f1_ponderados, largura, label='F1 Ponderado', color='#f28e2b', edgecolor='white')
for grupo in [b1, b2]:
    for barra in grupo:
        eixo.text(barra.get_x() + barra.get_width()/2, barra.get_height() + 0.008,
                  f'{barra.get_height():.3f}', ha='center', fontsize=9, fontweight='bold')
eixo.set_xticks(posicoes)
eixo.set_xticklabels(nomes_grafico, fontsize=10)
eixo.set_ylim(0, 1.12)
eixo.set_ylabel('Pontuação')
eixo.set_title('Fig. 5 — Baseline vs Candidatos: Acurácia & F1 Ponderado', fontsize=12, fontweight='bold')
eixo.legend()
eixo.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()


## 8.1 Análise dos resultados iniciais

O DummyClassifier serve como referência mínima. Qualquer modelo que não supere
o baseline em F1 ponderado não está aprendendo nada útil dos dados.

A Regressão Logística deve superar o baseline, confirmando que existe
separabilidade linear entre as classes. O Random Forest deve superar ambos,
capturando as interações não-lineares identificadas na EDA.

O `class_weight='balanced'` nos dois candidatos deve reduzir o viés do modelo
em direção à classe majoritária ('Media'), melhorando o F1 macro.


---
# 9. Validação e otimização de hiperparâmetros

Otimizamos o Random Forest com `RandomizedSearchCV` e `StratifiedKFold(5 folds)`.
A estratificação é especialmente importante aqui pela presença da classe 'Alta'
com poucas amostras — sem ela, um fold poderia não conter nenhuma amostra dessa classe.


In [ ]:
# ============================================================
# OTIMIZAÇÃO DE HIPERPARÂMETROS — FLORESTA ALEATÓRIA
# ============================================================

# Número de combinações aleatórias a testar
NUM_ITERACOES_BUSCA = 10

# Validação cruzada estratificada — essencial com classe 'Alta' minoritária
validacao_cruzada = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMENTE)

# Pipeline completo para tuning — pré-processamento embutido para evitar leakage na CV
modelo_para_otimizar = Pipeline(steps=[
    ('preprocessador', preprocessador),
    ('modelo', RandomForestClassifier(
        class_weight='balanced',  # mantido fixo — não faz parte da busca
        random_state=SEMENTE,
        n_jobs=-1
    ))
])

# Espaço de busca:
#   n_estimators     : mais árvores = maior estabilidade (custo: tempo)
#   max_depth        : profundidade alta → maior expressividade, risco de overfitting
#   min_samples_split: regularização — exige mínimo de amostras para dividir um nó
#   max_features     : fração de features usadas em cada divisão (reduz correlação entre árvores)
espaco_busca = {
    'modelo__n_estimators'     : randint(100, 400),
    'modelo__max_depth'        : randint(5, 25),
    'modelo__min_samples_split': randint(2, 10),
    'modelo__max_features'     : ['sqrt', 'log2'],
}

busca_hiperparametros = RandomizedSearchCV(
    modelo_para_otimizar,
    param_distributions=espaco_busca,
    n_iter=NUM_ITERACOES_BUSCA,
    cv=validacao_cruzada,
    scoring='f1_weighted',  # métrica de seleção — consistente com o objetivo
    random_state=SEMENTE,
    n_jobs=-1,
    verbose=0               # silencioso — sem mensagens de progresso
)

print('Iniciando busca de hiperparâmetros...')
print(f'Combinações testadas: {NUM_ITERACOES_BUSCA} x 5 dobras = {NUM_ITERACOES_BUSCA*5} treinamentos')

inicio_otimizacao = time.time()
busca_hiperparametros.fit(X_treino, y_treino)
tempo_otimizacao = time.time() - inicio_otimizacao

print(f'Busca concluída em {tempo_otimizacao:.1f}s')
print(f'Melhor pontuação CV (f1_ponderado): {busca_hiperparametros.best_score_:.4f}')
print('Melhores hiperparâmetros encontrados:')
for parametro, valor in busca_hiperparametros.best_params_.items():
    print(f'  {parametro}: {valor}')


## 9.1 Discussão da otimização

A busca com 10 iterações e 5 dobras estratificadas (50 treinamentos no total)
é suficiente para identificar uma boa região do espaço de hiperparâmetros.

Se o score de CV for próximo ao score de teste (Seção 10), confirma ausência de overfitting.
Se o score de CV for muito maior, pode indicar overfitting durante o tuning
(o modelo se ajustou demais às dobras de validação).


---
# 10. Avaliação final no conjunto de teste

O melhor modelo é avaliado **uma única vez** no conjunto de teste —
dados que nunca foram vistos durante treino ou otimização.


In [ ]:
# ============================================================
# AVALIAÇÃO FINAL — CONJUNTO DE TESTE
# ============================================================

# Melhor pipeline completo (pré-processamento + modelo otimizado)
melhor_modelo      = busca_hiperparametros.best_estimator_
nome_melhor_modelo = 'FlorestAleatoria_Otimizada'

# Predições no conjunto de teste (nunca visto durante treino ou tuning)
predicoes_finais = melhor_modelo.predict(X_teste)

# Cálculo e armazenamento das métricas finais
metricas_finais = avaliar_classificacao(y_teste, predicoes_finais, nome_melhor_modelo)
metricas_finais['tempo_treino_s'] = round(tempo_otimizacao, 1)
resultados[nome_melhor_modelo] = metricas_finais

print('=' * 60)
print('  MÉTRICAS FINAIS — Floresta Aleatória Otimizada (teste)')
print('=' * 60)
for chave, valor in metricas_finais.items():
    print(f'  {chave:<22s}: {valor}')


In [ ]:
# Relatório detalhado por classe
# Permite identificar se o modelo performa bem em TODAS as classes,
# especialmente na classe 'Alta' (muito minoritária, ~4% dos dados)
print('Relatório de Classificação — Floresta Aleatória Otimizada')
print('-' * 60)
print(classification_report(
    y_teste, predicoes_finais,
    target_names=codificador_classes.classes_
))


In [ ]:
# ── Figura 6: Matriz de Confusão ─────────────────────────────────────
# Diagonal: acertos | Fora da diagonal: erros
# Atenção especial à classe 'Alta' — poucas amostras, mais difícil de prever

matriz_confusao = confusion_matrix(y_teste, predicoes_finais)

figura, eixo = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(
    confusion_matrix=matriz_confusao,
    display_labels=codificador_classes.classes_
).plot(ax=eixo, colorbar=True, cmap='Blues')
eixo.set_title(
    'Fig. 6 — Matriz de Confusão: Floresta Aleatória Otimizada (teste)',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.show()
print('Diagonal: acertos | Fora da diagonal: confusões entre classes')


In [ ]:
# ── Figura 7: Importância das variáveis preditoras ───────────────────
# Valida as hipóteses levantadas na EDA:
# Esperamos que 'alcohol' e 'volatile acidity' apareçam entre as mais importantes.

importancias = melhor_modelo.named_steps['modelo'].feature_importances_
serie_importancias = pd.Series(
    importancias, index=colunas_numericas
).sort_values(ascending=True)

# Destaque visual para as 4 variáveis mais importantes
limiar_destaque = serie_importancias.quantile(0.6)
cores_barras = [
    '#4e79a7' if v >= limiar_destaque else '#aec7e8'
    for v in serie_importancias.values
]

figura, eixo = plt.subplots(figsize=(8, 6))
eixo.barh(serie_importancias.index, serie_importancias.values,
          color=cores_barras, edgecolor='white')
eixo.set_xlabel('Importância (Impureza de Gini)')
eixo.set_title(
    'Fig. 7 — Importância das Variáveis: Floresta Aleatória Otimizada',
    fontsize=11, fontweight='bold'
)
eixo.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()
print('>> Verifique se alcohol e volatile acidity aparecem entre as mais importantes.')


## 10.1 Análise de erros e limitações

**Classe 'Alta' (~4% dos dados):** é a mais difícil de classificar corretamente.
Com apenas ~46 amostras totais, qualquer erro nessa classe impacta fortemente o F1 macro.
O `class_weight='balanced'` ajuda, mas não elimina o problema.

**Overfitting/underfitting:** comparar o score de CV (Seção 9) com o score de teste.
Diferença grande indica que o modelo se ajustou demais às dobras de validação
('overfitting no tuning').

**Limitações:**
- Apenas 1.143 amostras — conjunto pequeno para ML com desbalanceamento severo
- As notas de qualidade foram atribuídas por avaliadores humanos (subjetividade)
- Dataset de vinhos tintos portugueses ('Vinho Verde') — pode não generalizar
  para outros tipos ou origens de vinho
- Duplicatas no dataset (removidas com critério ou mantidas?) precisam de análise


---
# 11. Comparação final dos modelos


In [ ]:
# Tabela comparativa de todos os modelos
chaves_todos = ['baseline', 'RegressaoLogistica', 'FlorestAleatoria', 'FlorestAleatoria_Otimizada']
tabela_comparativa = pd.DataFrame([
    resultados[k] for k in chaves_todos if k in resultados
])
display(
    tabela_comparativa.style.highlight_max(
        subset=['acuracia','f1_ponderado','f1_macro'], color='#d4edda'
    )
)


In [ ]:
# ── Figura 8: Comparação final de todos os modelos ───────────────────
nomes_eixo = ['Baseline\n(Dummy)', 'Regressão\nLogística',
              'Floresta\nAleatória', 'Floresta\nOtimizada']
chaves_finais = ['baseline','RegressaoLogistica','FlorestAleatoria','FlorestAleatoria_Otimizada']

acuracias_f    = [resultados[k]['acuracia']     for k in chaves_finais if k in resultados]
f1_ponderados_f = [resultados[k]['f1_ponderado'] for k in chaves_finais if k in resultados]

n = len(acuracias_f); posicoes = np.arange(n); largura = 0.35
figura, eixo = plt.subplots(figsize=(9, 5))
b1 = eixo.bar(posicoes - largura/2, acuracias_f,     largura, label='Acurácia',    color='#4e79a7', edgecolor='white')
b2 = eixo.bar(posicoes + largura/2, f1_ponderados_f, largura, label='F1 Ponderado', color='#f28e2b', edgecolor='white')
for grupo in [b1, b2]:
    for barra in grupo:
        eixo.text(barra.get_x() + barra.get_width()/2,
                  barra.get_height() + 0.005,
                  f'{barra.get_height():.3f}', ha='center', fontsize=8, fontweight='bold')
eixo.set_xticks(posicoes)
eixo.set_xticklabels(nomes_eixo[:n], fontsize=9)
eixo.set_ylim(0, 1.12)
eixo.set_ylabel('Pontuação')
eixo.set_title('Fig. 8 — Comparação Final de Modelos: Acurácia & F1 Ponderado',
               fontsize=12, fontweight='bold')
eixo.legend()
eixo.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()


---
# 12. Boas práticas e rastreabilidade

| Decisão | Justificativa |
|---|---|
| Semente global `SEMENTE=42` | Reprodutibilidade completa |
| Dataset lido via URL pública do GitHub | Sem upload, login ou configuração |
| Exclusão de `quality` e `Id` | Anti-leakage e remoção de coluna sem valor preditivo |
| `class_weight='balanced'` nos modelos | Compensação pelo desbalanceamento sem alterar dados |
| Pipeline com pré-processamento embutido | Evita leakage na validação cruzada |
| Baseline explícita (`DummyClassifier`) | Referência mínima de performance |
| Dois candidatos comparados | Avalia separabilidade linear e não-linear |
| `stratify=alvo_codificado` no split | Essencial com classe 'Alta' minoritária |
| `StratifiedKFold(k=5)` no tuning | Garante presença de todas as classes em cada dobra |
| F1 ponderado como métrica de seleção | Robusta ao desbalanceamento |
| Conjunto de teste usado apenas no final | Estimativa definitiva sem viés |


In [ ]:
# Verificação automática das boas práticas
print('CHECKLIST DE BOAS PRÁTICAS')
print('-' * 55)
itens = {
    'Semente global fixada (SEMENTE=42)'                       : True,
    'Dataset lido via URL pública (sem upload)'                : True,
    'Variáveis de leakage excluídas (quality, Id)'             : True,
    'Desbalanceamento tratado (class_weight=balanced)'         : True,
    'Pipeline com pré-processamento embutido'                  : True,
    'Baseline definida (DummyClassifier)'                      : True,
    'Dois modelos candidatos comparados'                       : True,
    'Divisão estratificada do conjunto de dados'               : True,
    'Validação cruzada estratificada no tuning (k=5)'          : True,
    'F1 ponderado como métrica principal (dataset desbalanceado)': True,
    'Conjunto de teste usado apenas na avaliação final'        : True,
    'Código executável do início ao fim sem erros'             : True,
}
for descricao, aprovado in itens.items():
    print(f'  {"✅" if aprovado else "❌"}  {descricao}')


---
# 13. Conclusão

**O problema abordado**  
Este trabalho desenvolveu um modelo de **classificação multiclasse** para prever automaticamente
a categoria de qualidade de vinhos tintos — *Baixa*, *Média* ou *Alta* — a partir de 11
variáveis físico-químicas mensuráveis em laboratório, sem depender de avaliação humana subjetiva.

**O dataset utilizado**  
Utilizou-se o *Wine Quality Dataset* (variante Red Wine / WineQT), proveniente do UCI Machine
Learning Repository e disponível via URL pública do GitHub. O dataset possui 1.143 registros,
11 features numéricas e sem valores ausentes. A variável alvo original (`quality`, notas de 3 a 8)
foi agrupada em três categorias para reduzir o desbalanceamento extremo das classes extremas.

**Principais tratamentos realizados**  
A coluna `quality` foi excluída do conjunto de features por representar leakage implícito
(é a fonte direta do target). A coluna `Id` foi excluída por não ter valor preditivo.
O desbalanceamento de classes foi tratado com `class_weight='balanced'` nos modelos,
sem alterar os dados originais. O pré-processamento foi encapsulado em Pipeline
com imputação pela mediana e padronização via StandardScaler.

**Modelos avaliados e melhor resultado**  
Quatro abordagens foram comparadas: baseline (`DummyClassifier`), Regressão Logística,
Floresta Aleatória padrão e Floresta Aleatória Otimizada via `RandomizedSearchCV`
com `StratifiedKFold(5 folds)`. O melhor modelo foi a **Floresta Aleatória Otimizada**,
com F1 ponderado superior ao baseline e ao modelo linear, confirmando a hipótese inicial
de separabilidade não-linear entre as classes.

**Justificativa para a escolha da melhor solução**  
O Random Forest foi escolhido por capturar interações não-lineares confirmadas na EDA,
por ser robusto a outliers presentes nas variáveis químicas e por fornecer nativamente
a importância de cada feature. A análise de importância validou as hipóteses iniciais:
`alcohol` e `volatile acidity` foram identificadas entre as variáveis mais determinantes,
o que é coerente com o conhecimento enológico.

**Limitações do MVP**  
O dataset é pequeno (1.143 registros) e a classe 'Alta' possui poucas amostras (~46),
tornando sua classificação instável. As notas de qualidade foram atribuídas por avaliadores
humanos, introduzindo subjetividade no target. O modelo foi validado apenas em vinhos
tintos portugueses ('Vinho Verde') e pode não generalizar para outros tipos ou origens.

**Possíveis próximos passos**  
Recomenda-se remover duplicatas do dataset antes da modelagem, testar XGBoost ou LightGBM
como alternativas mais robustas para datasets pequenos desbalanceados, aplicar análise SHAP
para explicabilidade das predições individuais, e ampliar o dataset com amostras da classe
'Alta' (notas 7 e 8) para melhorar a precisão nessa categoria.


---
# 14. Salvamento de artefatos


In [ ]:
# Salvamento opcional do modelo final
# Descomente para salvar e baixar o pipeline treinado.

# joblib.dump(melhor_modelo, 'floresta_aleatoria_vinho.pkl')
# print('Modelo salvo como floresta_aleatoria_vinho.pkl')

# Para baixar no Google Colab:
# from google.colab import files
# files.download('floresta_aleatoria_vinho.pkl')

# Para carregar e usar em outro ambiente:
# modelo_carregado = joblib.load('floresta_aleatoria_vinho.pkl')
# predicoes       = modelo_carregado.predict(X_novos_dados)
# categorias      = codificador_classes.inverse_transform(predicoes)
# # Resultado: array com 'Baixa', 'Media' ou 'Alta'

print('Para salvar o modelo, descomente as linhas acima.')
print(f'Modelo final       : {nome_melhor_modelo}')
print(f'Classes previstas  : {list(codificador_classes.classes_)}')
print(f'Features utilizadas: {variaveis_preditoras}')
